In [1]:
import pandas as pd
import xarray as xr
import numpy as np
import os.path

from metpy.calc import mixing_ratio_from_relative_humidity,specific_humidity_from_mixing_ratio
from metpy.units import units

data_dir = '../data_out/'
stats_dir = '../stats_csv/'

In [2]:
def get_df_from_nc(file, var, loc_sel=None):
    ds = xr.open_dataset(file)
    df = pd.DataFrame(ds[var].values, columns=ds.location, index=pd.to_datetime(ds.time,utc=True))[dt_from:dt_to]
    if loc_sel is None:
        return df
    else:
        return df[loc_sel]

## Script settings

In [3]:
scope ='H2' # campaign: 'H2' or 'H3'

res = '3h'

In [4]:
# do scope specific settings
if scope == 'H2':
    scope_name = 'HEFEX II'

    loc_on = ['A260', 'D262', 'A267R', 'A267', 'A267L', 'D269',
              'D273', 'T275', 'D281', 'D285', 'D293',
              'T303', 'D305', 'A309', 'D316']
    loc_on_sel = ['T275','T303']
    loc_off = ['STHE']

    dt_from,dt_to = '2023-08-17', '2023-09-07 10:00'

    file_obs  = f'HEFEX2__Obs_AWS_L1__1h_avg30min_20230816-20230910.nc'
    file_icon = f'HEFEX2__ICON_v370_2030_aws_locations__1h_avg30min_20230815-20230914.nc'
elif scope == 'H3':
    scope_name = 'HEFEX III'

    loc_on = ['T272','T303']
    loc_on_sel = loc_on
    loc_off = ['STHE','IHE','A244']

    dt_from,dt_to = '2025-08-06', '2025-08-31 23:59'

    file_obs  = f'HEFEX3__Obs_AWS_L1__1h_avg30min_20250805-20250903.nc'
    file_icon = f'HEFEX3__ICON_v370_2030_aws_locations__1h_avg30min_20250805-20250905.nc'
else:
    print('Unknown scope!')

## Get data (ICON & Obs) and resample to daily

The data used for this figure is available at Zenodo:

HEFEX II: [![DOI](https://zenodo.org/badge/DOI/10.5281/zenodo.19568035.svg)](https://doi.org/10.5281/zenodo.19568035)

HEFEX III: [![DOI](https://zenodo.org/badge/DOI/10.5281/zenodo.19596032.svg)](https://doi.org/10.5281/zenodo.19596032)

In [5]:
df_temp_2m = get_df_from_nc(data_dir + file_obs, f't_2m')
df_temp_4m = get_df_from_nc(data_dir + file_obs, f't_4m')
df_rh_2m   = get_df_from_nc(data_dir + file_obs, f'rh_2m')
df_rh_4m   = get_df_from_nc(data_dir + file_obs, f'rh_4m')
df_pres    = get_df_from_nc(data_dir + file_obs, f'pres')

df_i_temp = get_df_from_nc(data_dir + file_icon, f't_5m') - 273.15
df_i_qv   = get_df_from_nc(data_dir + file_icon, f'qv_5m') * 1000

In [6]:
ds = xr.open_dataset(data_dir + file_obs)
df_ele = pd.DataFrame(ds.ele.values, columns=['obs_ele'], index=ds.location)

ds = xr.open_dataset(data_dir + file_icon)
df_ele = df_ele.join(pd.DataFrame(ds.ele.values, columns=['icon_ele'], index=ds.location), how='inner')

df_ele['delta'] = df_ele['icon_ele'] - df_ele['obs_ele']

In [7]:
def get_qv_from_rh(df_rh, df_temp, df_pres):
    # filter RH data_in and calculate QV
    df_rh[df_rh <= 5] = np.nan

    df_w = mixing_ratio_from_relative_humidity(df_pres[df_rh.columns].values * units.hPa, df_temp[df_rh.columns].values * units.degC, df_rh.values/100)
    df_qv = pd.DataFrame(specific_humidity_from_mixing_ratio(df_w).magnitude * 1000, columns=df_rh.columns, index=df_rh.index)
    return df_qv

df_qv_2m = get_qv_from_rh(df_rh_2m, df_temp_2m, df_pres)
df_qv_4m = get_qv_from_rh(df_rh_4m, df_temp_4m, df_pres)

In [8]:
def get_diff_matrix(df_icon, df_obs):
    loc = list(set(df_icon.columns) & set(df_obs.columns))

    df_obs_res = df_obs.resample(res).mean()
    # apply mask of NaN values in Obs to ICON data and resample
    df_icon_res = df_icon.mask(df_obs.isna()).resample(res).mean()
    return df_icon_res[loc] - df_obs_res[loc]

def get_stats(df_diff, loc_filter=None):
    if loc_filter is not None:
        loc = list(set(df_diff.columns) & set(loc_filter))
        df_diff = df_diff[loc]

    count = df_diff.count().sum()
    if count > 0:
        mean = np.nanmean(df_diff.values).round(2)
        std = np.nanstd(df_diff.values).round(2)
    else:
        mean, std = np.nan, np.nan
    stats = {'loc': df_diff.columns.to_list(),
             'mean': mean,
             'std': std,
             'count': count,
             'dz': df_ele.loc[df_diff.columns,'delta'].mean().round(2)}
    stats['formatted'] = f"{stats['mean']} ± {stats['std']} (n={stats['count']}, dz={stats['dz']}m)"
    return stats

def merge_stats(stats_temp, stats_qv):
    stats = {}
    if stats_temp['count'] == stats_qv['count']:
        stats['n'] = stats_temp['count']
    else:
        stats['n'] = f"{stats_temp['count']} / {stats_qv['count']}"
    stats['dz'] = stats_temp['dz']
    if stats_temp['count'] > 0:
        stats['dT'] = f"{stats_temp['mean']} ± {stats_temp['std']}"
    else:
        stats['dT'] = 'NA'
    if stats_qv['count'] > 0:
        stats['dq'] = f"{stats_qv['mean']} ± {stats_qv['std']}"
    else:
        stats['dq'] = 'NA'
    return stats

def print_stats(title, stats_temp, stats_qv):
    print(72*'-')
    print(title)
    print('Location(s) temp:', stats_temp['loc'])
    print('Temp (mean ± std):', stats_temp['formatted'])
    print('Location(s) qv:', stats_qv['loc'])
    print('qv (mean ± std):', stats_qv['formatted'])

In [9]:
df_dT_2m = get_diff_matrix(df_i_temp, df_temp_2m)
df_dT_4m = get_diff_matrix(df_i_temp, df_temp_4m)
df_dq_2m = get_diff_matrix(df_i_qv, df_qv_2m)
df_dq_4m = get_diff_matrix(df_i_qv, df_qv_4m)

print(scope_name)
print_stats('on-glacier (all), 2m', get_stats(df_dT_2m, loc_on), get_stats(df_dq_2m, loc_on))
if scope == 'H2':
    print_stats('on-glacier (selected), 2m', get_stats(df_dT_2m, loc_on_sel), get_stats(df_dq_2m, loc_on_sel))
    print_stats('on-glacier (selected), 4m', get_stats(df_dT_4m, loc_on_sel), get_stats(df_dq_4m, loc_on_sel))
elif scope == 'H3':
    print_stats('on-glacier (all), 4m', get_stats(df_dT_4m, loc_on), get_stats(df_dq_4m, loc_on))
print_stats('off-glacier (all), 2m', get_stats(df_dT_2m, loc_off), get_stats(df_dq_2m, loc_off))


HEFEX II
------------------------------------------------------------------------
on-glacier (all), 2m
Location(s) temp: ['D285', 'D305', 'D316', 'T303', 'D293', 'D262', 'A309', 'A267', 'A267L', 'A260', 'D269', 'T275', 'D281', 'A267R', 'D273']
Temp (mean ± std): 0.42 ± 1.65 (n=2528, dz=61.35m)
Location(s) qv: ['D285', 'D305', 'D316', 'T303', 'D293', 'D262', 'A309', 'A267', 'A267L', 'A260', 'D269', 'T275', 'D281', 'A267R', 'D273']
qv (mean ± std): 0.01 ± 0.87 (n=1186, dz=61.35m)
------------------------------------------------------------------------
on-glacier (selected), 2m
Location(s) temp: ['T275', 'T303']
Temp (mean ± std): 0.02 ± 1.48 (n=344, dz=45.17m)
Location(s) qv: ['T275', 'T303']
qv (mean ± std): 0.16 ± 0.92 (n=329, dz=45.17m)
------------------------------------------------------------------------
on-glacier (selected), 4m
Location(s) temp: ['T275', 'T303']
Temp (mean ± std): -0.34 ± 1.05 (n=329, dz=45.17m)
Location(s) qv: ['T275', 'T303']
qv (mean ± std): -0.73 ± 1.77 (n=3

In [10]:
df = pd.DataFrame()

df['on_glacier_all_2m'] = merge_stats(get_stats(df_dT_2m, loc_on), get_stats(df_dq_2m, loc_on))
if scope == 'H2':
    df['on_glacier_sel_2m'] = merge_stats(get_stats(df_dT_2m, loc_on_sel), get_stats(df_dq_2m, loc_on_sel))
    df['on_glacier_sel_4m'] = merge_stats(get_stats(df_dT_4m, loc_on_sel), get_stats(df_dq_4m, loc_on_sel))
elif scope == 'H3':
    df['on_glacier_all_4m'] = merge_stats(get_stats(df_dT_4m, loc_on), get_stats(df_dq_4m, loc_on))

df['off_glacier_all_2m'] = merge_stats(get_stats(df_dT_2m, loc_off), get_stats(df_dq_2m, loc_off))

out_file = stats_dir + f'HEFEX{scope[1]}__AWS_ICON_stats__res{res}.csv'
df.to_csv(out_file)
print('Saved stats to', out_file)
df

Saved stats to ../stats_csv/HEFEX2__AWS_ICON_stats__res3h.csv


,on_glacier_all_2m,on_glacier_sel_2m,on_glacier_sel_4m,off_glacier_all_2m
n,2528 / 1186,344 / 329,329,172 / 0
dz,61.35,45.17,45.17,7.57
dT,0.42 ± 1.65,0.02 ± 1.48,-0.34 ± 1.05,-0.47 ± 0.99
dq,0.01 ± 0.87,0.16 ± 0.92,-0.73 ± 1.77,NA


## Merge HEFEX II and III stats into one table for display in paper

In [11]:
h2_file = stats_dir + f'HEFEX2__AWS_ICON_stats__res{res}.csv'
h3_file = stats_dir + f'HEFEX3__AWS_ICON_stats__res{res}.csv'

if os.path.isfile(h2_file) and os.path.isfile(h3_file):
    print('Both files exist, proceeding with merge.')

    df_h2 = pd.read_csv(h2_file, index_col=0)
    df_h3 = pd.read_csv(h3_file, index_col=0)

    df_merged = df_h2.merge(df_h3, left_index=True, right_index=True, how='inner', suffixes=('_h2', '_h3'))
    display(df_merged)
else:
    print('One or both files are missing. Please check the file paths or generate them first.')
    df_merged = None


Both files exist, proceeding with merge.


,on_glacier_all_2m_h2,on_glacier_sel_2m,on_glacier_sel_4m,off_glacier_all_2m_h2,on_glacier_all_2m_h3,on_glacier_all_4m,off_glacier_all_2m_h3
n,2528 / 1186,344 / 329,329,172 / 0,411 / 405,411 / 197,510
dz,61.35,45.17,45.17,7.57,49.86,49.86,-1.89
dT,0.42 ± 1.65,0.02 ± 1.48,-0.34 ± 1.05,-0.47 ± 0.99,0.12 ± 1.24,-0.39 ± 1.06,-0.23 ± 1.26
dq,0.01 ± 0.87,0.16 ± 0.92,-0.73 ± 1.77,NaN,-0.02 ± 0.89,0.07 ± 0.86,-0.22 ± 0.97


In [12]:
if df_merged is not None:
    latex_table = df_merged.to_latex()

    latex_table = latex_table.replace(r'n ', r'\# of paired obs.$^a$ ($N$) ')

    latex_table = latex_table.replace(r'dz ', r'$\overline{\Delta z}$ in m ')
    latex_table = latex_table.replace(r'dT ', r'$\Delta T$ (mean ± 1$\sigma$) in K ')
    latex_table = latex_table.replace(r'dq ', r'$\Delta q$ (mean ± 1$\sigma$) in g kg$^{-1}$ ')

    print(latex_table)

\begin{tabular}{llllllll}
\toprule
 & on_glacier_all_2m_h2 & on_glacier_sel_2m & on_glacier_sel_4m & off_glacier_all_2m_h2 & on_glacier_all_2m_h3 & on_glacier_all_4m & off_glacier_all_2m_h3 \\
\midrule
\# of paired obs.$^a$ ($N$) & 2528 / 1186 & 344 / 329 & 329 & 172 / 0 & 411 / 405 & 411 / 197 & 510 \\
$\overline{\Delta z}$ in m & 61.35 & 45.17 & 45.17 & 7.57 & 49.86 & 49.86 & -1.89 \\
$\Delta T$ (mean ± 1$\sigma$) in K & 0.42 ± 1.65 & 0.02 ± 1.48 & -0.34 ± 1.05 & -0.47 ± 0.99 & 0.12 ± 1.24 & -0.39 ± 1.06 & -0.23 ± 1.26 \\
$\Delta q$ (mean ± 1$\sigma$) in g kg$^{-1}$ & 0.01 ± 0.87 & 0.16 ± 0.92 & -0.73 ± 1.77 & NaN & -0.02 ± 0.89 & 0.07 ± 0.86 & -0.22 ± 0.97 \\
\bottomrule
\end{tabular}

